### Evaluation: Numerical Drift vs. Inherent Architecture Error (Baseline Spatial UNet)

**Objective:** 
To isolate and quantify the exact cause of error in the baseline NeuralMAG UNet model, distinguishing between its inherent mathematical mapping limitations and the accumulated temporal drift caused by its Markovian (memoryless) architecture.

**Methodology (Clean History Mapping):**
This script initializes a random $64 \times 64$ magnetization state and relaxes it to a 5-vortex ground state. It then executes two parallel LLG simulations:
1. **Film 1 (Ground Truth):** Driven exclusively by the exact Fast Fourier Transform (FFT) solver.
2. **Film 2 (Surrogate):** Driven by the UNet's predicted demagnetizing field.

At every time step, we track two distinct error metrics:
*   **Instantaneous Mapping Error (Blue Line):** We feed the UNet the *clean, perfect history* from Film 1 and compare its prediction to the FFT. This proves the inherent mathematical accuracy of the architecture.
*   **Accumulated Drift (Red Line):** We track the error of Film 2 as the UNet feeds its own corrupted predictions back into itself over thousands of steps. 

**Expected Result:** 
The gap between the Instantaneous Error and the Accumulated Drift mathematically justifies the necessity of upgrading to a non-Markovian (Derivative Buffer) architecture to stabilize long-term temporal predictions.


In [ ]:
# Initialize global constants
MODEL_NAME = "baseline_original_test"

from google.colab import drive
drive.mount('/content/drive')

In [2]:
%cd /content/drive/MyDrive/NeuralMAG_Data/egs/NMI/vortex_evaluate

In [4]:
import sys
import os

# Append the root directory where the 'libs' folder lives
# Update this path if 'libs' is inside a different project root folder
project_root = "/content/drive/MyDrive/NeuralMAG_Data"
if project_root not in sys.path:
    sys.path.append(project_root)

print(project_root)
print("Python search path updated successfully!")

In [7]:
#model_name = "model_original"
base_path = "/content/drive/MyDrive/NeuralMAG_Data/NeuralMAG_Models"
#base_path = "/home/username/NeuralMAG_Data" # for VCU cluster


# run compute_vortex.py to compare model with FFT simulation and track error
!PYTHONPATH=/content/drive/MyDrive/NeuralMAG_Data CUDA_VISIBLE_DEVICES=0 python compute_vortex.py \
  --gpu 0 \
  --krn 16 \
  --w 64 \
  --layers 2 \
  --InitCore 5 \
  --base_path {base_path} \
  --model_name {MODEL_NAME} \
  --ckpt_path {base_path}/{MODEL_NAME}_folder/{MODEL_NAME}.pt

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# 1. Define your current run details to pinpoint the files
BASE_PATH = "/content/drive/MyDrive/NeuralMAG_Data/NeuralMAG_Models"
#MODEL_NAME = "baseline_original_unet"
KERNEL_SIZE = 16
GRID_SIZE = 64
INIT_CORE = 5
SPLIT = 32
RAND_SEED = 10000 # The specific seed folder you want to visualize
# POTENTIALLY ADD AVERAGING OF SEEDS FOR A SINGLE AVERAGE PLOT

# 2. Reconstruct the clean nested directory layout
results_dir = f'{BASE_PATH}/{MODEL_NAME}_folder/eval_results/{MODEL_NAME}/k{KERNEL_SIZE}_size{GRID_SIZE}_InitCore{INIT_CORE}/split{SPLIT}_rand{RAND_SEED}'

drift_path = os.path.join(results_dir, 'error_film2_drift.npy')
inst_path = os.path.join(results_dir, 'error_film1_inst.npy')

if not os.path.exists(drift_path):
    print(f"No evaluation arrays found under:\n{results_dir}")
else:
    drift_arr = np.load(drift_path)
    inst_arr = np.load(inst_path)
    
    steps = drift_arr[0, :]
    drift_error = drift_arr[1, :]
    inst_error = inst_arr[1, :]

    # Plot creation
    plt.figure(figsize=(10, 5))
    plt.plot(steps, drift_error, color='red', linewidth=1.7, label='Accumulated Drift (film2 - corrupted feedback)')
    plt.plot(steps, inst_error, color='blue', linewidth=1.7, label='Instantaneous Mapping (film1 - clean FFT history)')
    
    plt.title(f'Numerical Drift vs. Inherent Architecture Error\nModel: {MODEL_NAME} | Seed: {RAND_SEED}', fontsize=12, fontweight='bold')
    plt.xlabel('Simulation Steps')
    plt.ylabel('Error (Relative to FFT Ground Truth)')
    plt.legend(loc='upper left')
    plt.grid(True, linestyle='--', alpha=0.5)
    
    # Save chart image cleanly right alongside the data files
    plot_save_path = os.path.join(results_dir, 'drift_vs_error_chart.png')
    plt.savefig(plot_save_path, dpi=300, bbox_inches='tight')
    print(f"Error plot saved to google drive location: {plot_save_path}")
    files.download(plot_save_path)
    plt.show()